# Momentum on the CAC 40 (Colab version)

To run everything: Runtime menu, then Run all. It takes a few minutes to download the prices.

In [ ]:
!pip install -q -U yfinance

In [ ]:
import os
os.makedirs('src', exist_ok=True)
os.makedirs('data', exist_ok=True)
open('src/__init__.py', 'w').close()

In [ ]:
%%writefile src/data.py
"""Loading CAC 40 prices from Yahoo Finance."""
from pathlib import Path

import numpy as np
import pandas as pd

CAC40_TICKERS = [
    "AI.PA", "AIR.PA", "ALO.PA", "MT.AS", "CS.PA", "BNP.PA", "EN.PA",
    "CAP.PA", "CA.PA", "ACA.PA", "BN.PA", "DSY.PA", "ENGI.PA", "EL.PA",
    "ERF.PA", "RMS.PA", "KER.PA", "OR.PA", "LR.PA", "MC.PA", "ML.PA",
    "ORA.PA", "RI.PA", "PUB.PA", "RNO.PA", "SAF.PA", "SGO.PA", "SAN.PA",
    "SU.PA", "GLE.PA", "STLAP.PA", "STMPA.PA", "TEP.PA", "HO.PA", "TTE.PA",
    "URW.AS", "VIE.PA", "DG.PA", "WLN.PA", "VIV.PA",
]
BENCHMARK = "^FCHI"


def download_prices(tickers, start="2005-01-01", end=None, cache="data/prices.csv"):
    """Download adjusted prices and keep them in a CSV file."""
    cache = Path(cache)
    if cache.exists():
        return pd.read_csv(cache, index_col=0, parse_dates=True)
    import yfinance as yf

    raw = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False)
    prices = raw["Close"].dropna(axis=1, how="all")
    missing = sorted(set(tickers) - set(prices.columns))
    if missing:
        print(f"No data for {missing}")
    cache.parent.mkdir(parents=True, exist_ok=True)
    prices.to_csv(cache)
    return prices


def to_monthly(prices):
    """Last price of each month, leaving out the current month if it is not over."""
    monthly = prices.resample("ME").last()
    if prices.index[-1] < monthly.index[-1] - pd.Timedelta(days=3):
        monthly = monthly.iloc[:-1]
    return monthly


def synthetic_prices(n_assets=40, years=15, seed=0):
    """Random prices to test the code offline. Not meant to say anything about markets."""
    rng = np.random.default_rng(seed)
    idx = pd.bdate_range("2008-01-01", periods=years * 252)
    market = rng.normal(0.0003, 0.01, len(idx))
    rets = market[:, None] + rng.normal(0.0001, 0.015, (len(idx), n_assets))
    cols = [f"S{i:02d}" for i in range(n_assets)]
    return pd.DataFrame(100 * np.exp(np.cumsum(rets, axis=0)), index=idx, columns=cols)

In [ ]:
%%writefile src/strategy.py
"""Cross-sectional long/short momentum, after Jegadeesh and Titman (1993)."""
import numpy as np
import pandas as pd


def momentum_signal(monthly_prices, lookback=12, skip=1):
    """Return between t-lookback and t-skip, using only prices known at the end of month t."""
    return monthly_prices.shift(skip) / monthly_prices.shift(lookback) - 1


def long_short_weights(signal, quantile=0.2):
    """Buy the top quantile and short the bottom one, equal weights, summing to zero."""
    w = pd.DataFrame(0.0, index=signal.index, columns=signal.columns)
    for date, row in signal.iterrows():
        row = row.dropna()
        n = int(len(row) * quantile)
        if n < 1:
            continue
        ranked = row.sort_values()
        w.loc[date, ranked.index[-n:]] = 1.0 / n
        w.loc[date, ranked.index[:n]] = -1.0 / n
    return w


def backtest(monthly_prices, lookback=12, skip=1, quantile=0.2, cost_bps=10.0):
    """Monthly strategy returns, before and after costs.

    Weights chosen at the end of month t are applied to the return of month t+1.
    Costs are proportional to turnover. The borrowing cost of short positions
    is not included.
    """
    rets = monthly_prices.pct_change()
    w = long_short_weights(momentum_signal(monthly_prices, lookback, skip), quantile)
    w_held = w.shift(1)
    gross = (w_held * rets).sum(axis=1, min_count=1)
    turnover = w_held.diff().abs().sum(axis=1)
    net = gross - turnover * cost_bps / 1e4
    out = pd.DataFrame({"gross": gross, "net": net, "turnover": turnover})
    return out.dropna(subset=["gross"]).loc[lambda d: d.index >= w_held.dropna(how="all").index[1]]

In [ ]:
%%writefile src/metrics.py
"""Performance measures for monthly returns."""
import numpy as np
import pandas as pd
import statsmodels.api as sm

PERIODS = 12


def annual_return(r):
    return (1 + r).prod() ** (PERIODS / len(r)) - 1


def annual_vol(r):
    return r.std() * np.sqrt(PERIODS)


def sharpe(r, rf=0.0):
    """Annualized Sharpe ratio, with a zero risk-free rate by default."""
    ex = r - rf / PERIODS
    return ex.mean() / ex.std() * np.sqrt(PERIODS)


def max_drawdown(r):
    wealth = (1 + r).cumprod()
    return (wealth / wealth.cummax() - 1).min()


def alpha_beta(r, benchmark):
    """Regress r on the benchmark (Newey-West errors). Alpha is annualized."""
    df = pd.concat([r, benchmark], axis=1).dropna()
    X = sm.add_constant(df.iloc[:, 1])
    res = sm.OLS(df.iloc[:, 0], X).fit(cov_type="HAC", cov_kwds={"maxlags": 3})
    return {
        "alpha_annual": res.params.iloc[0] * PERIODS,
        "alpha_tstat": res.tvalues.iloc[0],
        "beta": res.params.iloc[1],
    }


def summary(r, name="strategy"):
    return pd.Series(
        {
            "Annualized return": annual_return(r),
            "Annualized volatility": annual_vol(r),
            "Sharpe": sharpe(r),
            "Max drawdown": max_drawdown(r),
        },
        name=name,
    )

# Momentum on the CAC 40

This notebook tests a momentum strategy on CAC 40 stocks. At the end of each month I rank the stocks by their return between t-12 and t-1, buy the top 20%, short the bottom 20% and hold the portfolio for one month. I want to know whether it still pays once transaction costs are paid.

In [ ]:
import sys; sys.path.insert(0, ".")
import os
import matplotlib.pyplot as plt
import pandas as pd

from src.data import CAC40_TICKERS, BENCHMARK, download_prices, to_monthly, synthetic_prices
from src.strategy import backtest, long_short_weights, momentum_signal
from src import metrics as m

plt.rcParams["figure.figsize"] = (9, 4)

## 1. Data

In [ ]:
if os.environ.get("DEMO"):
    prices = synthetic_prices()
else:
    prices = download_prices(CAC40_TICKERS + [BENCHMARK], start="2005-01-01", cache="data/prices.csv")

index_px = prices.pop(BENCHMARK) if BENCHMARK in prices else None
monthly = to_monthly(prices)
print(f"{monthly.shape[1]} stocks, {monthly.index[0]:%Y-%m} -> {monthly.index[-1]:%Y-%m}")
monthly.iloc[:, :5].tail()

## 2. Baseline backtest

12-1 window, quintiles, 10 basis points of costs. I compare it with an equal-weighted portfolio of the same stocks.

In [ ]:
res = backtest(monthly, lookback=12, skip=1, quantile=0.2, cost_bps=10)
bench_ew = monthly.pct_change().mean(axis=1).reindex(res.index)

fig, ax = plt.subplots()
(1 + res["gross"]).cumprod().plot(ax=ax, label="Momentum L/S (gross)")
(1 + res["net"]).cumprod().plot(ax=ax, label="Momentum L/S (net of costs)")
(1 + bench_ew).cumprod().plot(ax=ax, label="Equal-weighted (long only)", alpha=.7)
ax.set_title("Cumulative performance (base 1)"); ax.legend(); plt.show()

In [ ]:
table = pd.concat([
    m.summary(res["gross"], "Momentum gross"),
    m.summary(res["net"], "Momentum net"),
    m.summary(bench_ew, "Equal-weighted"),
], axis=1)
print(f"Average monthly turnover: {res['turnover'].mean():.2f}")
table.round(3)

### Worst months

I look at which stocks weighed the most in the months where the strategy loses the most, to see whether a drop comes from a bad price.

In [ ]:
w_held = long_short_weights(momentum_signal(monthly, 12, 1), 0.2).shift(1)
contrib = w_held * monthly.pct_change()
for d, r in res["net"].nsmallest(5).items():
    worst = contrib.loc[d].nsmallest(3)
    print(f"{d:%Y-%m}: {r:+.1%} | worst contributions: " + ", ".join(f"{t} {v:+.1%}" for t, v in worst.items()))

## 3. Alpha and beta

I regress the net strategy returns on the equal-weighted portfolio returns, with Newey-West standard errors.

In [ ]:
pd.DataFrame({"vs equal-weighted": m.alpha_beta(res["net"], bench_ew)}).round(3)

## 4. Robustness

Does the result hold when the formation window or the cost level changes?

In [ ]:
rows = {}
for lb in [3, 6, 9, 12]:
    r = backtest(monthly, lookback=lb)["net"]
    rows[f"{lb}-month window"] = {"Ann. return": m.annual_return(r), "Sharpe": m.sharpe(r), "Max DD": m.max_drawdown(r)}
pd.DataFrame(rows).T.round(3)

In [ ]:
rows = {}
for c in [0, 10, 20, 50]:
    r = backtest(monthly, cost_bps=c)["net"]
    rows[f"{c} bps"] = {"Ann. return": m.annual_return(r), "Sharpe": m.sharpe(r)}
pd.DataFrame(rows).T.round(3)

## 5. Out-of-sample test

I pick the formation window on the first 60% of the months, then look at what that choice gives on the rest of the period.

In [ ]:
split = monthly.index[int(len(monthly) * 0.6)]
train_sharpe = {lb: m.sharpe(backtest(monthly, lookback=lb)["net"].loc[:split]) for lb in [3, 6, 9, 12]}
best = max(train_sharpe, key=train_sharpe.get)
test = backtest(monthly, lookback=best)["net"].loc[split:]
print(f"Split: {split:%Y-%m} | best window (train): {best} months")
print(f"Train Sharpe: {train_sharpe[best]:.2f} | Test Sharpe: {m.sharpe(test):.2f}")

## 6. Limitations

- The stocks are the current members of the CAC 40. Companies that left the index are not in the sample.
- With about forty stocks, each quintile holds around 8 names, so the results are noisy.
- Costs are proportional to turnover. The borrowing cost of short positions is ignored.
- The risk-free rate is taken as zero in the Sharpe ratio.
- I did not use the Fama-French factors, which would be a natural next step.